# 02 — KPI Engineering & Composite Score

**AI Equity Research Lab** — FGV EAESP (Aula 3)

This notebook:
1. Runs the KPI computation pipeline
2. Runs the composite score computation
3. Visualizes: KPI heatmap, score ranking, correlation matrix
4. Prints Top 3 and Bottom 3 with justifications

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)
%matplotlib inline

print(f"Project root: {PROJECT_ROOT}")

## 1. Run KPI Pipeline

In [ ]:
from src.features.kpis import run as run_kpis

kpis = run_kpis()

In [ ]:
kpis = pd.read_parquet(PROCESSED_DIR / "kpis_wide.parquet")
print(f"Shape: {kpis.shape}")
print(f"Dtypes:\n{kpis.dtypes}")
print(f"\nYears: {sorted(kpis['year'].unique())}")
print(f"Tickers: {sorted(kpis['ticker'].unique())}")
kpis.head(10)

## 2. Run Composite Score

In [ ]:
from src.features.score import run as run_scores

scores = run_scores()

In [ ]:
scores = pd.read_parquet(PROCESSED_DIR / "scores_2024.parquet")
print(f"Shape: {scores.shape}")
scores

---
## 3. KPI Heatmap (2024, normalized 0–1)

In [ ]:
kpis_2024 = kpis[kpis["year"] == 2024].copy()

kpi_cols = [
    "net_margin", "roe", "roa", "ebit_margin",
    "debt_to_equity", "net_debt_to_ebit",
    "current_ratio", "cash_to_revenue",
    "revenue_cagr", "net_income_cagr",
    "volatility_annualized", "max_drawdown",
    "momentum_6m", "momentum_12m",
]

# Normalize each column to 0-1 (min-max)
heatmap_data = kpis_2024.set_index("ticker")[kpi_cols].copy()
for col in heatmap_data.columns:
    col_min = heatmap_data[col].min()
    col_max = heatmap_data[col].max()
    if col_max != col_min:
        heatmap_data[col] = (heatmap_data[col] - col_min) / (col_max - col_min)
    else:
        heatmap_data[col] = 0.5

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    mask=heatmap_data.isna(),
    cbar_kws={"label": "Normalized (0=worst, 1=best)"},
)
ax.set_title("KPI Heatmap — 2024 (min-max normalized)", fontsize=14)
ax.set_ylabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Composite Score Ranking

In [ ]:
scores_sorted = scores.sort_values("composite_score", ascending=True)

# Color by sector
sector_colors = {
    "Bancos": "#2563eb",
    "Energia El\u00e9trica": "#16a34a",
}
colors = [sector_colors.get(s, "#888") for s in scores_sorted["sector"]]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    scores_sorted["ticker"],
    scores_sorted["composite_score"],
    color=colors,
    edgecolor="white",
    linewidth=0.5,
)

# Add score labels
for bar, score in zip(bars, scores_sorted["composite_score"]):
    ax.text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
        f"{score:.1f}", va="center", fontsize=10, fontweight="bold",
    )

ax.set_xlabel("Composite Score (0-100)")
ax.set_title("Composite Fundamental Score — 2024 Ranking", fontsize=14)
ax.set_xlim(0, 80)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2563eb", label="Bancos"),
    Patch(facecolor="#16a34a", label="Energia El\u00e9trica"),
]
ax.legend(handles=legend_elements, loc="lower right")

plt.tight_layout()
plt.show()

## 5. Top 3 & Bottom 3 Justifications

In [ ]:
kpis_2024 = kpis[kpis["year"] == 2024].set_index("ticker")
scores_ranked = scores.sort_values("rank_overall")

def justify(ticker, rank, score):
    """Generate a one-line justification based on the ticker's KPIs."""
    k = kpis_2024.loc[ticker] if ticker in kpis_2024.index else None
    parts = []
    if k is not None:
        if not pd.isna(k.get("roe")):
            parts.append(f"ROE {k['roe']:.1%}")
        if not pd.isna(k.get("net_margin")):
            parts.append(f"net margin {k['net_margin']:.1%}")
        if not pd.isna(k.get("revenue_cagr")):
            parts.append(f"revenue CAGR {k['revenue_cagr']:.1%}")
        if not pd.isna(k.get("volatility_annualized")):
            parts.append(f"vol {k['volatility_annualized']:.1%}")
        if not pd.isna(k.get("debt_to_equity")):
            parts.append(f"D/E {k['debt_to_equity']:.2f}x")
        if not pd.isna(k.get("momentum_12m")):
            parts.append(f"12m momentum {k['momentum_12m']:.1%}")
    detail = ", ".join(parts) if parts else "limited data available"
    return f"#{rank} {ticker} — score {score:.1f}: {detail}"

print("=" * 70)
print("TOP 3")
print("=" * 70)
for _, row in scores_ranked.head(3).iterrows():
    print(justify(row["ticker"], int(row["rank_overall"]), row["composite_score"]))

print()
print("=" * 70)
print("BOTTOM 3")
print("=" * 70)
for _, row in scores_ranked.tail(3).iterrows():
    print(justify(row["ticker"], int(row["rank_overall"]), row["composite_score"]))

## 6. KPI Correlation Matrix

In [ ]:
kpi_cols = [
    "net_margin", "roe", "roa", "ebit_margin",
    "debt_to_equity", "net_debt_to_ebit",
    "current_ratio", "cash_to_revenue",
    "revenue_cagr", "net_income_cagr",
    "volatility_annualized", "max_drawdown",
    "momentum_6m", "momentum_12m",
]

corr = kpis[kpi_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("KPI Correlation Matrix (all years)", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## 7. Data Quality Check

In [ ]:
print("=" * 60)
print("NaN SUMMARY — kpis_wide (2024)")
print("=" * 60)

k24 = kpis[kpis["year"] == 2024].set_index("ticker")
nan_count = k24[kpi_cols].isna().sum(axis=1)
nan_pct = k24[kpi_cols].isna().mean(axis=1) * 100

for ticker in k24.index:
    n = nan_count[ticker]
    p = nan_pct[ticker]
    flag = " <<<" if p > 30 else ""
    print(f"  {ticker:8s}  {n:2d}/{len(kpi_cols)} NaN ({p:5.1f}%){flag}")

print("\n" + "=" * 60)
print("NaN SUMMARY — scores_2024")
print("=" * 60)
score_nan = scores.isna().sum()
for col, n in score_nan.items():
    if n > 0:
        print(f"  {col}: {n} NaN")
if score_nan.sum() == 0:
    print("  No NaN in scores table.")